In [1]:
from google.colab import drive
import pandas as pd
import numpy as np
import torchaudio
import zipfile
import time
import math
import ast
import os

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F

from sklearn.model_selection import KFold

In [2]:
import pandas as pd
import numpy as np
import torchaudio
import torch

print(f"pandas version: {pd.__version__}")
print(f"numpy version: {np.__version__}")
print(f"torchaudio version: {torchaudio.__version__}")
print(f"torch version: {torch.__version__}")

pandas version: 2.2.2
numpy version: 2.0.2
torchaudio version: 2.11.0+cu128
torch version: 2.11.0+cu128


In [3]:
import sys
print(f"Python version: {sys.version}")

Python version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [ ]:
drive.mount('/content/drive')

zip_path = "/content/drive/MyDrive/dataset.zip"
extract_path = "/content/dataset/"

os.makedirs(extract_path, exist_ok=True)

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

model_save_path = "/content/drive/MyDrive/SSL_Projesi/1DCNN/"

test_csv = '/content/drive/MyDrive/SSL_Projesi/test_metadata.csv'
test_df = pd.read_csv(test_csv)

train_csv = '/content/drive/MyDrive/SSL_Projesi/train_metadata.csv'
train_df = pd.read_csv(train_csv)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Mounted at /content/drive


In [ ]:
class FullCache:
  def __init__(self, df, preload=True):
    self.df = df.reset_index(drop=True)
    self.preload = preload
    self.waves = []
    self.targets = []
    self.angles = []

    if preload:
        print(f"{len(self.df)} dosya RAM'e yükleniyor...", flush=True)
        for idx, row in self.df.iterrows():
            wav, _ = torchaudio.load(row['audio_path'])

            az = float(row['azimuth_deg'])
            rad = math.radians(az)

            target = torch.tensor([
                math.sin(rad),
                math.cos(rad)
            ], dtype=torch.float32)

            angle = torch.tensor(az, dtype=torch.float32)

            self.waves.append(wav)
            self.targets.append(target)
            self.angles.append(angle)
        print(f"Preload tamamlandı. ({len(self.df)})", flush=True)

dataset_cache = FullCache(train_df)

6480 dosya RAM'e yükleniyor...
Preload tamamlandı. (6480)


In [ ]:
class IndexDataset(Dataset):
  def __init__(self, cache, indices):
    self.cache = cache
    self.indices = indices

  def __len__(self):
    return len(self.indices)

  def __getitem__(self, idx):
    idx = self.indices[idx]

    return (
        self.cache.waves[idx],
        self.cache.targets[idx],
        self.cache.angles[idx]
    )


In [ ]:
def create_kfold_loaders(cache, batch_size=256, K=5, preload=False):

    kf = KFold(n_splits=K, shuffle=True, random_state=42)

    nw = min(2, os.cpu_count() or 1)

    folds = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(range(len(cache.df)))):

        print(f"\n[FOLD {fold+1}/{K}]")

        train_dataset = IndexDataset(cache, train_idx)
        val_dataset   = IndexDataset(cache, val_idx)

        train_loader = DataLoader(
            train_dataset,
            batch_size=batch_size,
            shuffle=True,
            num_workers=nw,
            pin_memory=True,
            drop_last=True
        )

        val_loader = DataLoader(
            val_dataset,
            batch_size=batch_size,
            shuffle=False,
            num_workers=nw,
            pin_memory=True
        )

        folds.append((train_loader, val_loader))

    return folds

folds = create_kfold_loaders(dataset_cache, batch_size=256, K=5, preload=True)


[FOLD 1/5]

[FOLD 2/5]

[FOLD 3/5]

[FOLD 4/5]

[FOLD 5/5]


In [ ]:
class SSLCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(8, 32, kernel_size=1, padding=0),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 32, kernel_size=9, dilation=1, padding=4),
            nn.BatchNorm1d(32),
            nn.ReLU(),

            nn.Conv1d(32, 64, kernel_size=9, dilation=2, padding=8),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            nn.Conv1d(64, 128, kernel_size=9, dilation=4, padding=16),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            ## 128 -> 128 di sen bu hale getirdin dilation 4 tü padding 16
            nn.Conv1d(128, 256, kernel_size=9, dilation=8, padding=32),
            nn.BatchNorm1d(256),
            nn.ReLU(),
        )

        self.attention = nn.Sequential(
            nn.Conv1d(256, 128, kernel_size=1),
            nn.ReLU(),
            nn.Conv1d(128, 1, kernel_size=1)
        )

        self.head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x, T=0.5):
        features = self.features(x)

        scores = self.attention(features)
        weights = torch.softmax(scores / T, dim=-1)
        pooled = (weights * features).sum(dim=-1)

        out = self.head(pooled)
        out = F.normalize(out, dim=-1)
        return out

In [ ]:
def angular_error_deg(pred_sin_cos: torch.Tensor,
                      true_angles_deg: torch.Tensor) -> float:
    pred_deg = torch.rad2deg(
        torch.atan2(pred_sin_cos[:, 0], pred_sin_cos[:, 1])
    )
    pred_deg = (pred_deg + 360.0) % 360.0
    diff = torch.abs(pred_deg - true_angles_deg)
    return torch.min(diff, 360.0 - diff).mean().item()

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, delta=0.1, path='best_ssl_model.pth'):
        self.patience  = patience
        self.delta     = delta
        self.path      = path
        self.counter   = 0
        self.best_err  = float('inf')
        self.early_stop = False

    def __call__(self, val_error_deg: float, model: nn.Module):
        if val_error_deg < self.best_err - self.delta:
            self.best_err = val_error_deg
            torch.save(model.state_dict(), self.path)
            self.counter = 0
        else:
            self.counter += 1
            print(f"    [EarlyStopping] Sayaç: {self.counter}/{self.patience} "
                  f"(en iyi: {self.best_err:.2f}°)")
            if self.counter >= self.patience:
                self.early_stop = True

In [ ]:
NUM_EPOCHS = 100

history = []

for fold, (train_loader, val_loader) in enumerate(folds):

    total_start = time.time()

    print(f"\n================ FOLD {fold+1} ================")

    # model her fold'da SIFIRLANIR
    model = SSLCNN().to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
    criterion = torch.nn.MSELoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=2
    )

    current_fold_path = f"best_model_fold_{fold+1}.pth"
    current_fold_path = os.path.join(model_save_path,current_fold_path)

    early_stopping = EarlyStopping(patience=15, delta=0.01, path=current_fold_path)

    for epoch in range(1, NUM_EPOCHS + 1):

        ep_start = time.time()

        # ---------------- TRAIN ----------------
        model.train()
        train_loss, train_err = 0, 0

        for wav, targets, angle in train_loader:
            wav, targets, angle = wav.to(device), targets.to(device), angle.to(device)

            optimizer.zero_grad()

            pred = model(wav)
            loss = criterion(pred, targets)

            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 0.5)

            optimizer.step()

            train_loss += loss.item()
            train_err += angular_error_deg(pred, angle)

        avg_train_err = train_err / len(train_loader)
        avg_train_loss = train_loss / len(train_loader)

        # ---------------- VAL ----------------
        model.eval()
        val_loss, val_err = 0, 0

        with torch.no_grad():
            for wav, targets, angle in val_loader:
                wav, targets, angle = wav.to(device), targets.to(device), angle.to(device)

                pred = model(wav)
                loss = criterion(pred, targets)

                val_loss += loss.item()
                val_err += angular_error_deg(pred, angle)

        avg_val_err = val_err / len(val_loader)
        avg_val_loss = val_loss / len(val_loader)

        ep_dur  = time.time() - ep_start
        ep_m, ep_s = divmod(int(ep_dur), 60)

        elapsed      = time.time() - total_start
        eta_sec      = (elapsed / epoch) * (NUM_EPOCHS - epoch)
        eta_m, eta_s = divmod(int(eta_sec), 60)

        cur_lr = optimizer.param_groups[0]['lr']

        history.append({
            'fold': fold + 1,
            'epoch': epoch,
            'train_loss': avg_train_loss,
            'train_err': avg_train_err,
            'val_loss': avg_val_loss,
            'val_err': avg_val_err,
            'lr': cur_lr
        })

        print(
            f"Fold {fold+1} | Epoch {epoch} | "
            f"LR: {cur_lr:.2e} | "
            f"Train → Loss: {avg_train_loss:.4f}  Hata: {avg_train_err:.1f}° | "
            f"Val → Loss: {avg_val_loss:.4f}  Hata: {avg_val_err:.1f}° | "
            f"ETA: {eta_m}m{eta_s:02d}s"
        )

        scheduler.step(avg_val_err)

        early_stopping(avg_val_err, model)
        if early_stopping.early_stop:
            print("\n[!] Early Stopping: eğitim durduruldu.")
            break


    df_history = pd.DataFrame(history)
    csv_path = os.path.join(model_save_path, f"history_fold_{fold+1}.csv")
    df_history.to_csv(csv_path, index=False)

    print(f"--- Fold {fold+1} kayıtları kaydedildi: {csv_path}")



================ FOLD 1 ================
Fold 1 | Epoch 1 | LR: 3.00e-04 | Train → Loss: 0.6719  Hata: 66.2° | Val → Loss: 0.9891  Hata: 89.5° | ETA: 18m17s
Fold 1 | Epoch 2 | LR: 3.00e-04 | Train → Loss: 0.3223  Hata: 39.1° | Val → Loss: 0.4655  Hata: 50.0° | ETA: 16m11s
Fold 1 | Epoch 3 | LR: 3.00e-04 | Train → Loss: 0.1680  Hata: 25.2° | Val → Loss: 0.2052  Hata: 28.5° | ETA: 15m10s
Fold 1 | Epoch 4 | LR: 3.00e-04 | Train → Loss: 0.1206  Hata: 20.2° | Val → Loss: 0.3779  Hata: 43.5° | ETA: 14m36s
    [EarlyStopping] Sayaç: 1/15 (en iyi: 28.53°)
Fold 1 | Epoch 5 | LR: 3.00e-04 | Train → Loss: 0.1094  Hata: 19.7° | Val → Loss: 0.0984  Hata: 19.0° | ETA: 14m12s
Fold 1 | Epoch 6 | LR: 3.00e-04 | Train → Loss: 0.0896  Hata: 17.6° | Val → Loss: 0.1965  Hata: 29.0° | ETA: 13m53s
    [EarlyStopping] Sayaç: 1/15 (en iyi: 18.98°)
Fold 1 | Epoch 7 | LR: 3.00e-04 | Train → Loss: 0.0816  Hata: 16.3° | Val → Loss: 0.0929  Hata: 17.9° | ETA: 13m37s
Fold 1 | Epoch 8 | LR: 3.00e-04 | Train → Loss: 